# Granite Speech 4.1 2B - Local Inference

Multilingual ASR and speech translation using [ibm-granite/granite-speech-4.1-2b](https://huggingface.co/ibm-granite/granite-speech-4.1-2b).

**Capabilities:**
- Automatic Speech Recognition (English, French, German, Spanish, Portuguese, Japanese)
- Bidirectional speech translation (to/from English)
- Keyword biasing for names, acronyms, and jargon
- Punctuation and capitalization

## Setup

In [25]:
# Install PyTorch with CUDA 12.8 support (required for GPU acceleration)
%pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu128

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\Kurt Valcorza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [26]:
%pip install transformers>=4.52.1 soundfile huggingface_hub

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: C:\Users\Kurt Valcorza\AppData\Local\Python\pythoncore-3.14-64\python.exe -m pip install --upgrade pip


In [27]:
import torch
import soundfile as sf
import numpy as np
from huggingface_hub import hf_hub_download
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

Using device: cuda
GPU: NVIDIA GeForce RTX 5070 Ti Laptop GPU
VRAM: 11.9 GB


## Load Model

In [28]:
model_name = "ibm-granite/granite-speech-4.1-2b"

processor = AutoProcessor.from_pretrained(model_name)
tokenizer = processor.tokenizer

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_name, torch_dtype=torch.bfloat16
).to(device)
print("Model loaded.")

Loading weights: 100%|██████████| 954/954 [00:00<00:00, 10898.30it/s]


Model loaded.


## Load Audio

Using the sample audio bundled with the model. Replace `audio_path` with your own `.wav` file (mono, 16kHz).

In [29]:
# Download sample audio from the model repo
audio_path = hf_hub_download(repo_id=model_name, filename="multilingual_sample.wav")

# Or use your own file:
# audio_path = "path/to/your/audio.wav"

data, sr = sf.read(audio_path, dtype='float32')

# Convert to mono if stereo
if data.ndim > 1:
    data = data.mean(axis=1)

# Resample to 16kHz if needed
if sr != 16000:
    import torchaudio.functional as F
    wav = torch.from_numpy(data).unsqueeze(0)
    wav = F.resample(wav, sr, 16000)
    sr = 16000
else:
    wav = torch.from_numpy(data).unsqueeze(0)

print(f"Audio: {wav.shape[1] / sr:.1f}s, {sr} Hz, mono")

Audio: 24.9s, 16000 Hz, mono


## Transcribe (ASR with Punctuation)

In [30]:
user_prompt = "<|audio|>transcribe the speech with proper punctuation and capitalization."

chat = [{"role": "user", "content": user_prompt}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
model_outputs = model.generate(**model_inputs, max_new_tokens=200, do_sample=False, num_beams=1)

num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
output_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)

print("Transcription:")
print(output_text[0])

Transcription:
For Timothy was a spoiled cat, and he allowed no one to interfere. Everybody waited upon him, moving their chairs even, for he was monarch of the hearth. Dinarzade, la nuit suivante, appela sa soeur quand il en fut temps. "Si vous ne dormez pas, ma soeur," lui dit-elle, "je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur."


## Transcribe (Raw - No Punctuation)

In [31]:
user_prompt = "<|audio|>can you transcribe the speech into a written format?"

chat = [{"role": "user", "content": user_prompt}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
model_outputs = model.generate(**model_inputs, max_new_tokens=200, do_sample=False, num_beams=1)

num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
output_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)

print("Raw transcription:")
print(output_text[0])

Raw transcription:
for timothy was a spoiled cat, and he allowed no one to interfere. everybody waited upon him, moving their chairs even, for he was monarch of the hearth. dinarzade la nuit suivante appela sa soeur quand il en fut temps. "si vous ne dormez pas, ma soeur," lui dit-elle, "je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur."


## Translate Speech to Another Language

In [32]:
# Supported target languages: English, French, German, Spanish, Japanese, Italian, Mandarin
target_language = "Spanish"

user_prompt = f"<|audio|>translate the speech to {target_language} with proper punctuation and capitalization."

chat = [{"role": "user", "content": user_prompt}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
model_outputs = model.generate(**model_inputs, max_new_tokens=200, do_sample=False, num_beams=1)

num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
output_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)

print(f"Translation ({target_language}):")
print(output_text[0])

Translation (Spanish):
Porque Timothy era un gato empañado, y no le permitía que nadie interfiriera. Todo el mundo se ocupaba de él, moviendo sus sillas, incluso, porque él era monarca del hogar. Dinarzade la noche siguiente llamó a su hermano cuando era hora de dormir. "Si no duermeis, hermano," dijo a su hermano, "yo os pido en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur."


## Keyword Biasing

Improve recognition of specific names, acronyms, or technical terms by providing a keyword list.

In [33]:
keywords = ["Granite", "IBM", "ASR"]
keywords_str = ", ".join(keywords)

user_prompt = f"<|audio|>transcribe the speech to text. Keywords: {keywords_str}"

chat = [{"role": "user", "content": user_prompt}]
prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
model_outputs = model.generate(**model_inputs, max_new_tokens=200, do_sample=False, num_beams=1)

num_input_tokens = model_inputs["input_ids"].shape[-1]
new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
output_text = tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)

print(f"Transcription (biased towards: {keywords_str}):")
print(output_text[0])

Transcription (biased towards: Granite, IBM, ASR):
for timothy was a spoiled cat, and he allowed no one to interfere. everybody waited upon him, moving their chairs even, for he was monarch of the hearth. dinarzade la nuit suivante appela sa soeur quand il en fut temps. "si vous ne dormez pas, ma soeur," lui dit-elle, "je vous prie en attendant le jour qui paraîtra bientôt de continuer le conte du pêcheur."


## Helper Function

Reusable function for quick inference on any audio file.

In [34]:
def transcribe(audio_path, task="transcribe", target_language=None, keywords=None, punctuate=True):
    """
    Transcribe or translate an audio file.

    Args:
        audio_path: Path to audio file
        task: 'transcribe' or 'translate'
        target_language: Target language for translation (English, French, German, Spanish, Japanese, Italian, Mandarin)
        keywords: List of keywords for biasing
        punctuate: Whether to include punctuation and capitalization

    Returns:
        Transcribed/translated text
    """
    wav, sr = sf.read(audio_path, dtype='float32')
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    if sr != 16000:
        import torchaudio.functional as F
        wav = F.resample(torch.from_numpy(wav).unsqueeze(0), sr, 16000)
    else:
        wav = torch.from_numpy(wav).unsqueeze(0)

    if task == "translate" and target_language:
        if punctuate:
            instruction = f"translate the speech to {target_language} with proper punctuation and capitalization."
        else:
            instruction = f"translate the speech to {target_language}."
    elif keywords:
        keywords_str = ", ".join(keywords)
        instruction = f"transcribe the speech to text. Keywords: {keywords_str}"
    elif punctuate:
        instruction = "transcribe the speech with proper punctuation and capitalization."
    else:
        instruction = "can you transcribe the speech into a written format?"

    user_prompt = f"<|audio|>{instruction}"
    chat = [{"role": "user", "content": user_prompt}]
    prompt = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)

    model_inputs = processor(prompt, wav, device=device, return_tensors="pt").to(device)
    model_outputs = model.generate(**model_inputs, max_new_tokens=200, do_sample=False, num_beams=1)

    num_input_tokens = model_inputs["input_ids"].shape[-1]
    new_tokens = model_outputs[0, num_input_tokens:].unsqueeze(0)
    return tokenizer.batch_decode(new_tokens, add_special_tokens=False, skip_special_tokens=True)[0]


# Example usage:
# result = transcribe("my_audio.wav")
# result = transcribe("my_audio.wav", task="translate", target_language="French")
# result = transcribe("my_audio.wav", keywords=["PyTorch", "CUDA"])